# 후쿠오카 숙박업소 리뷰 — 감성분석 파인튜닝 v2 (의사 레이블링 포함)

## 실행 전 확인
- **메뉴 → 런타임 → 런타임 유형 변경 → GPU(T4)** 선택 필수
- 총 소요 시간: 약 40~60분

## 실행 순서
1. [1단계] 패키지 설치
2. [2단계] 기존 파일 정리
3. [3단계] 파일 업로드
4. [4단계] 데이터 준비 (의사 레이블링 포함)
5. [5단계] 베이스라인 F1 측정
6. [6단계] 파인튜닝
7. [7단계] 파인튜닝 후 F1 측정 및 비교
8. [8단계] 전체 81,613건 재분석
9. [9단계] 결과 다운로드

## [1단계] 패키지 설치
처음 한 번만 실행합니다.

In [ ]:
!pip install -q transformers datasets torch scikit-learn pandas
print('✅ 패키지 설치 완료')

## [2단계] 기존 파일 정리
이전 세션에서 남은 CSV 파일을 삭제합니다. **반드시 [3단계] 전에 실행하세요.**

In [ ]:
import os, glob
removed = []
for f in glob.glob('/content/*.csv'):
    os.remove(f)
    removed.append(f)
if removed:
    print('삭제된 파일:')
    for f in removed: print(f'  {f}')
else:
    print('삭제할 파일 없음')
print('✅ 완료 — 이제 [3단계]를 실행하세요.')

## [3단계] 파일 업로드
아래 3개 파일을 업로드합니다:
1. `labeling_sample_fixed.csv` — 사람 레이블 459건
2. `full_cleaned.csv` — 전처리 완료 원본 (81,613건)
3. `final_result_v2.csv` — v2 모델 예측 결과 (의사 레이블용)

In [ ]:
from google.colab import files
print('3개 파일을 선택해서 업로드하세요.')
uploaded = files.upload()
print('\n업로드된 파일:', list(uploaded.keys()))

# 파일명 자동 감지 (번호가 붙어도 인식)
import glob as g
def find_file(pattern):
    matches = g.glob(f'/content/{pattern}')
    if not matches:
        raise FileNotFoundError(f'{pattern} 파일을 찾을 수 없습니다.')
    return sorted(matches)[0]

LABEL_FILE  = find_file('labeling_sample*.csv')
FULL_FILE   = find_file('full_cleaned*.csv')
V2_FILE     = find_file('final_result*.csv')

print(f'\n사용할 파일:')
print(f'  레이블링: {LABEL_FILE}')
print(f'  원본    : {FULL_FILE}')
print(f'  v2 결과 : {V2_FILE}')
print('✅ 파일 확인 완료')

## [4단계] 데이터 준비 (의사 레이블링)
- 사람 레이블 459건 + v2 모델 고신뢰도 예측(의사 레이블) 4,000건을 합쳐 파인튜닝합니다
- 테스트셋은 사람 레이블만 사용해서 평가 신뢰도를 유지합니다

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

LABEL2ID = {'부정': 0, '긍정': 1}
ID2LABEL = {0: '부정', 1: '긍정'}

# ── 1. 사람 레이블 로드 ──────────────────────────────────────
df_human = pd.read_csv(LABEL_FILE)

def assign_label(row):
    lv = str(row.get('label', '')).strip()
    if lv == '긍정': return '긍정'
    elif lv == '부정': return '부정'
    try:
        r = float(row.get('rating', None))
        if r in [1.0, 2.0]: return '부정'
        elif r in [4.0, 5.0]: return '긍정'
    except: pass
    return None

df_human['label'] = df_human.apply(assign_label, axis=1)
df_human = df_human[df_human['label'].notna()].copy()
df_human['label_id'] = df_human['label'].map(LABEL2ID)
df_human['source'] = 'human'
print(f'사람 레이블: {len(df_human)}건')
print(df_human['label'].value_counts())

# ── 2. 의사 레이블 생성 (v2 고신뢰도) ───────────────────────
df_v2 = pd.read_csv(V2_FILE)
THRESHOLD = 0.97
df_pseudo_all = df_v2[df_v2['sentiment_score'] >= THRESHOLD].copy()
df_pseudo_all = df_pseudo_all.rename(columns={'sentiment': 'label'})
df_pseudo_all['label_id'] = df_pseudo_all['label'].map(LABEL2ID)
df_pseudo_all['source'] = 'pseudo'

# 사람 레이블과 중복 제거 후 균형 샘플링
human_texts = set(df_human['text'].tolist())
df_pseudo_all = df_pseudo_all[~df_pseudo_all['text'].isin(human_texts)]
PSEUDO_PER_CLASS = 2000
pseudo_pos = df_pseudo_all[df_pseudo_all['label']=='긍정'].sample(
    min(PSEUDO_PER_CLASS, (df_pseudo_all['label']=='긍정').sum()), random_state=42)
pseudo_neg = df_pseudo_all[df_pseudo_all['label']=='부정'].sample(
    min(PSEUDO_PER_CLASS, (df_pseudo_all['label']=='부정').sum()), random_state=42)
df_pseudo = pd.concat([pseudo_pos, pseudo_neg])
print(f'\n의사 레이블 (score>={THRESHOLD}): {len(df_pseudo)}건')
print(df_pseudo['label'].value_counts())

# ── 3. 합치기 ────────────────────────────────────────────────
cols = ['text', 'label', 'label_id', 'source']
df_combined = pd.concat([df_human[cols], df_pseudo[cols]])\
    .sample(frac=1, random_state=42).reset_index(drop=True)
print(f'\n최종 학습 데이터: {len(df_combined)}건')
print(df_combined['label'].value_counts())

# ── 4. 분할 (테스트셋 = 사람 레이블만) ──────────────────────
df_test    = df_human.sample(frac=0.15, random_state=42)
df_train_val = df_combined[~df_combined['text'].isin(df_test['text'])].copy()
df_train, df_val = train_test_split(
    df_train_val, test_size=0.15, random_state=42,
    stratify=df_train_val['label_id'])
print(f'\n분할: Train {len(df_train)} / Val {len(df_val)} / Test {len(df_test)} (사람 레이블만)')

## [5단계] 베이스라인 F1 측정 (파인튜닝 전)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, f1_score
import numpy as np

MODEL_NAME = 'alsgyu/sentiment-analysis-fine-tuned-model'
ORIG_LABEL_MAP = {'LABEL_0': '부정', 'LABEL_1': '중립', 'LABEL_2': '긍정'}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {device}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
base_model.eval().to(device)

def predict_batch(texts, model, tok, label_map, batch_size=32):
    preds = []
    id2label = model.config.id2label
    for i in range(0, len(texts), batch_size):
        batch = [t if isinstance(t, str) and t.strip() else '내용 없음'
                 for t in texts[i:i+batch_size]]
        inputs = tok(batch, return_tensors='pt', truncation=True,
                     padding=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            probs = torch.softmax(model(**inputs).logits, dim=-1).cpu().numpy()
        for prob in probs:
            idx = int(np.argmax(prob))
            mapped = label_map.get(id2label[idx], id2label[idx])
            if mapped == '중립':
                pos_i = [k for k,v in id2label.items() if v=='LABEL_2']
                neg_i = [k for k,v in id2label.items() if v=='LABEL_0']
                mapped = '긍정' if (pos_i and neg_i and prob[pos_i[0]] >= prob[neg_i[0]]) else '부정'
            preds.append(mapped)
    return preds

test_texts  = df_test['text'].tolist()
test_labels = df_test['label'].tolist()

print('베이스라인 예측 중...')
base_preds = predict_batch(test_texts, base_model, tokenizer, ORIG_LABEL_MAP)

y_true = [1 if l=='긍정' else 0 for l in test_labels]
y_pred = [1 if l=='긍정' else 0 for l in base_preds]
base_f1 = f1_score(y_true, y_pred, average='macro')

print(f'\n=== 베이스라인 성능 (파인튜닝 전) ===')
print(classification_report(y_true, y_pred, target_names=['부정','긍정'], labels=[0,1]))
print(f'Macro F1: {base_f1:.4f}')

## [6단계] 파인튜닝 (약 15~25분)

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from datasets import Dataset
from sklearn.metrics import f1_score
import numpy as np

def make_dataset(df, tok, max_length=128):
    ds = Dataset.from_dict({
        'text':  df['text'].fillna('').tolist(),
        'label': df['label_id'].tolist(),
    })
    return ds.map(lambda b: tok(b['text'], truncation=True, max_length=max_length), batched=True)

ft_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_ds = make_dataset(df_train, ft_tokenizer)
val_ds   = make_dataset(df_val,   ft_tokenizer)
test_ds  = make_dataset(df_test,  ft_tokenizer)

ft_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label=ID2LABEL, label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'macro_f1': f1_score(labels, preds, average='macro')}

training_args = TrainingArguments(
    output_dir='./ft_model',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=20,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(ft_tokenizer),
)

print(f'학습 데이터: {len(train_ds)}건 / 검증: {len(val_ds)}건 / 테스트: {len(test_ds)}건')
print('파인튜닝 시작...')
trainer.train()
print('✅ 파인튜닝 완료')

## [7단계] 파인튜닝 후 F1 측정 및 비교

In [ ]:
from sklearn.metrics import classification_report, f1_score

ft_preds_raw  = trainer.predict(test_ds)
ft_pred_ids   = np.argmax(ft_preds_raw.predictions, axis=-1)
ft_pred_labels = [ID2LABEL[i] for i in ft_pred_ids]

y_true_ft = [1 if l=='긍정' else 0 for l in test_labels]
ft_f1 = f1_score(y_true_ft, ft_pred_ids, average='macro')

print('=== 파인튜닝 후 성능 ===')
print(classification_report(y_true_ft, ft_pred_ids, target_names=['부정','긍정'], labels=[0,1]))
print(f'Macro F1: {ft_f1:.4f}')

print('\n' + '='*50)
print('성능 비교')
print('='*50)
print(f'베이스라인 Macro F1 : {base_f1:.4f}')
print(f'파인튜닝   Macro F1 : {ft_f1:.4f}')
print(f'개선율              : {(ft_f1-base_f1)/base_f1*100:+.1f}%')
if ft_f1 > base_f1:
    print('✅ 성능 향상!')
else:
    print('⚠️  개선 미미 — 하이퍼파라미터 조정 필요')

## [8단계] 전체 81,613건 재분석 (약 20~30분)

In [ ]:
from tqdm.notebook import tqdm
import time

df_full = pd.read_csv(FULL_FILE)
print(f'전체 데이터: {len(df_full):,}건')

ft_model.eval().to(device)
BATCH_SIZE = 32
texts = df_full['text'].fillna('').tolist()
sentiments, scores, score_pos_list, score_neg_list = [], [], [], []

t0 = time.time()
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='재분석 중'):
    batch = [t if t.strip() else '내용 없음' for t in texts[i:i+BATCH_SIZE]]
    inputs = ft_tokenizer(batch, return_tensors='pt', truncation=True,
                          padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        probs = torch.softmax(ft_model(**inputs).logits, dim=-1).cpu().numpy()
    for prob in probs:
        idx = int(np.argmax(prob))
        sentiments.append(ID2LABEL[idx])
        scores.append(float(prob[idx]))
        score_neg_list.append(float(prob[0]))
        score_pos_list.append(float(prob[1]))

print(f'\n✅ 완료! 소요 시간: {(time.time()-t0)/60:.1f}분')

df_v3 = df_full.copy()
df_v3['sentiment']      = sentiments
df_v3['sentiment_score'] = scores
df_v3['score_positive']  = score_pos_list
df_v3['score_negative']  = score_neg_list

print('\n감성 분포 (v3 파인튜닝):')
print(df_v3['sentiment'].value_counts())
df_v3.to_csv('final_result_v3.csv', index=False, encoding='utf-8-sig')
print('\n저장 완료: final_result_v3.csv')

## [9단계] 결과 다운로드

In [ ]:
from google.colab import files

files.download('final_result_v2.csv')
print('✅ final_result_v2.csv 다운로드 시작')

print('\n=== 최종 요약 ===')
print(f'베이스라인 Macro F1 : {base_f1:.4f}')
print(f'파인튜닝   Macro F1 : {ft_f1:.4f}')
print(f'개선율              : {(ft_f1-base_f1)/base_f1*100:+.1f}%')
print(f'학습 데이터         : 사람 레이블 {len(df_human)}건 + 의사 레이블 {len(df_pseudo)}건')
print(f'재분석 건수         : {len(df_v3):,}건')